# Binder-Protein Analyzer

This notebook demonstrates the analysis of protein-binder complexes from PDB/CIF files.

**Features:**
- Load structures from AlphaFold, Boltz, or other prediction tools
- Identify residue contacts (4-6 Å) between protein (chain A) and binder (chain B)
- Generate sequence proximity visualizations
- Analyze ensembles of multiple models
- Export results and figures

In [ ]:
# Import the analyzer
from analyzer import BinderAnalyzer, analyze_multiple_structures
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

%matplotlib inline
plt.style.use('default')

## 1. Single Structure Analysis

Analyze a single PDB/CIF file. The analyzer assumes:
- Chain A = Protein
- Chain B = Binder

In [ ]:
# Example: Replace with your structure file path
# structure_file = "path/to/your/complex.pdb"

# For demonstration, we'll create a sample if no file exists
import os
from pathlib import Path

# Check for example files in the directory
example_files = list(Path('.').glob('*.pdb')) + list(Path('.').glob('*.cif'))

if example_files:
    structure_file = str(example_files[0])
    print(f"Found structure file: {structure_file}")
else:
    print("No PDB/CIF files found in current directory.")
    print("Please add a structure file or use the example below.")

### Create Sample Structure (for testing without real data)

In [ ]:
# Create a sample PDB file for demonstration
sample_pdb = """REMARK   1 SAMPLE PROTEIN-BINDER COMPLEX
CRYST1   50.000   50.000   50.000  90.00  90.00  90.00 P 1           1
MODEL        1
ATOM      1  N   ALA A   1      10.000  10.000  10.000  1.00 20.00           N
ATOM      2  CA  ALA A   1      11.458  10.000  10.000  1.00 20.00           C
ATOM      3  C   ALA A   1      11.458  11.458  10.000  1.00 20.00           C
ATOM      4  O   ALA A   1      11.458  12.458   9.000  1.00 20.00           O
ATOM      5  CB  ALA A   1      12.000   9.000  11.000  1.00 20.00           C
"""

# Generate a simple two-chain structure
def generate_sample_pdb(filename="sample_complex.pdb", n_res_a=50, n_res_b=30):
    """Generate a sample PDB with two chains in contact."""
    atom_num = 1
    res_names = ['ALA', 'VAL', 'LEU', 'ILE', 'PHE', 'TRP', 'TYR', 'MET', 
                 'GLY', 'SER', 'THR', 'CYS', 'PRO', 'ASN', 'GLN', 
                 'ASP', 'GLU', 'LYS', 'ARG', 'HIS']
    
    lines = ["REMARK   1 SAMPLE PROTEIN-BINDER COMPLEX",
             "CRYST1   80.000   80.000   80.000  90.00  90.00  90.00 P 1           1"]
    
    # Chain A (protein) - extended conformation
    for i in range(n_res_a):
        res_name = res_names[i % len(res_names)]
        x = i * 3.8  # ~3.8A per residue in extended chain
        y = 0.0
        z = 0.0
        
        # Simple CA-only for clarity
        lines.append(f"ATOM  {atom_num:5d}  CA  {res_name:3s} A{i+1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00 20.00           C")
        atom_num += 1
    
    # Chain B (binder) - positioned to create contacts with middle of chain A
    contact_center = n_res_a // 2
    for i in range(n_res_b):
        res_name = res_names[(i+5) % len(res_names)]
        # Create contacts with chain A around residue 25
        x = (contact_center - n_res_b//2 + i) * 3.8
        y = 5.0 + np.sin(i * 0.5) * 2  # Varying distance for visualization
        z = 3.0
        
        lines.append(f"ATOM  {atom_num:5d}  CA  {res_name:3s} B{i+1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00 20.00           C")
        atom_num += 1
    
    lines.append("END")
    
    with open(filename, 'w') as f:
        f.write('\n'.join(lines))
    
    return filename

# Generate sample file
sample_file = generate_sample_pdb()
print(f"Generated sample file: {sample_file}")

In [ ]:
# Initialize analyzer with sample structure
analyzer = BinderAnalyzer(sample_file, protein_chain='A', binder_chain='B')

# Calculate contacts with 6A threshold
analyzer.calculate_contacts(distance_threshold=6.0)

# Print summary
analyzer.print_summary()

## 2. Contact Analysis Results

In [ ]:
# Get contacts as DataFrame
contacts_df = analyzer.get_contact_dataframe()
print(f"Total contacts: {len(contacts_df)}")
contacts_df.head(10)

In [ ]:
# Contact statistics
contacts_df.groupby('protein_resname')['distance'].agg(['count', 'mean', 'min']).sort_values('count', ascending=False)

## 3. Sequence Proximity Visualization

Linear sequence plot showing proximity-based coloring:
- **Darker color** = closer contact
- **Lighter color** = more distant contact

In [ ]:
# Plot sequence proximity
fig = analyzer.plot_sequence_proximity(
    figsize=(14, 6),
    distance_range=(4.0, 8.0)  # Color scale from 4A (dark) to 8A (light)
)
plt.tight_layout()
plt.show()

## 4. Contact Map

2D scatter plot showing which residues are in contact.

In [ ]:
# Plot contact map
fig = analyzer.plot_contact_map(figsize=(10, 8), distance_threshold=8.0)
plt.tight_layout()
plt.show()

## 5. Multi-Model Ensemble Analysis

Generate multiple models and analyze ensemble statistics.

In [ ]:
# Generate ensemble of structures with slight variations
def generate_ensemble(n_models=5, base_name="ensemble_model"):
    """Generate ensemble with varying binder positions."""
    files = []
    n_res_a, n_res_b = 50, 30
    res_names = ['ALA', 'VAL', 'LEU', 'ILE', 'PHE', 'TRP', 'TYR', 'MET', 
                 'GLY', 'SER', 'THR', 'CYS', 'PRO', 'ASN', 'GLN', 
                 'ASP', 'GLU', 'LYS', 'ARG', 'HIS']
    
    for model in range(n_models):
        filename = f"{base_name}_{model+1}.pdb"
        atom_num = 1
        lines = ["REMARK   1 ENSEMBLE MODEL",
                 "CRYST1   80.000   80.000   80.000  90.00  90.00  90.00 P 1           1"]
        
        # Chain A - fixed
        for i in range(n_res_a):
            res_name = res_names[i % len(res_names)]
            x = i * 3.8
            lines.append(f"ATOM  {atom_num:5d}  CA  {res_name:3s} A{i+1:4d}    {x:8.3f}{0.0:8.3f}{0.0:8.3f}  1.00 20.00           C")
            atom_num += 1
        
        # Chain B - varying position
        contact_center = n_res_a // 2 + np.random.randint(-5, 5)
        base_y = 5.0 + np.random.randn() * 0.5
        
        for i in range(n_res_b):
            res_name = res_names[(i+5) % len(res_names)]
            x = (contact_center - n_res_b//2 + i) * 3.8
            y = base_y + np.sin(i * 0.5) * (2 + np.random.randn() * 0.3)
            z = 3.0 + np.random.randn() * 0.5
            lines.append(f"ATOM  {atom_num:5d}  CA  {res_name:3s} B{i+1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}  1.00 20.00           C")
            atom_num += 1
        
        lines.append("END")
        
        with open(filename, 'w') as f:
            f.write('\n'.join(lines))
        files.append(filename)
    
    return files

ensemble_files = generate_ensemble(n_models=5)
print(f"Generated {len(ensemble_files)} ensemble models")
ensemble_files[:3]

In [ ]:
# Analyze ensemble (combine into single file with multiple models)
def combine_models(files, output="ensemble.pdb"):
    """Combine multiple PDB files into one with multiple models."""
    with open(output, 'w') as out:
        for i, f in enumerate(files):
            out.write(f"MODEL     {i+1}\n")
            with open(f) as inp:
                for line in inp:
                    if not line.startswith(('REMARK', 'CRYST1', 'END')):
                        out.write(line)
            out.write("ENDMDL\n")
        out.write("END\n")
    return output

ensemble_file = combine_models(ensemble_files)
print(f"Combined ensemble: {ensemble_file}")

In [ ]:
# Analyze the ensemble
ensemble_analyzer = BinderAnalyzer(ensemble_file)
ensemble_analyzer.calculate_contacts(distance_threshold=7.0)

# Calculate ensemble summary
summary_df = ensemble_analyzer.calculate_ensemble_summary()
print(f"Unique contacts across ensemble: {len(summary_df)}")
summary_df.head(15)

In [ ]:
# Ensemble summary statistics
print("\nContact Frequency Distribution:")
summary_df['contact_frequency'].describe()

In [ ]:
# Plot ensemble heatmap
fig = ensemble_analyzer.plot_ensemble_heatmap(figsize=(12, 10))
plt.tight_layout()
plt.show()

In [ ]:
# Plot ensemble-averaged sequence proximity
fig = ensemble_analyzer.plot_sequence_proximity(
    model_id=None,  # Use ensemble average
    figsize=(14, 6),
    distance_range=(4.0, 8.0)
)
plt.tight_layout()
plt.show()

## 6. Export Results

In [ ]:
# Create output directory
import os
os.makedirs('outputs', exist_ok=True)

# Save contact tables
ensemble_analyzer.save_contact_table('outputs/ensemble_contacts.csv')
summary_df.to_csv('outputs/ensemble_summary.csv', index=False)

# Save figures
ensemble_analyzer.plot_sequence_proximity(
    save_path='outputs/sequence_proximity.png'
)
plt.close()

ensemble_analyzer.plot_ensemble_heatmap(
    save_path='outputs/ensemble_heatmap.png'
)
plt.close()

print("Results saved to outputs/")

## 7. Multiple Structure Comparison

Compare multiple structure files.

In [ ]:
# Compare ensemble members
comparison = analyze_multiple_structures(ensemble_files, distance_threshold=7.0)
comparison

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(comparison['file'], comparison['epitope_residues'], color='coral')
axes[0].set_ylabel('Epitope Residues')
axes[0].set_title('Protein Epitope Size')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(comparison['file'], comparison['paratope_residues'], color='skyblue')
axes[1].set_ylabel('Paratope Residues')
axes[1].set_title('Binder Paratope Size')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Working with Real Data

Example of using the analyzer with actual AlphaFold/Boltz outputs:

In [ ]:
# # Uncomment and modify for your actual data:
# 
# # Single file analysis
# analyzer = BinderAnalyzer(
#     "path/to/alphafold_output.pdb",
#     protein_chain='A',
#     binder_chain='B'
# )
# 
# # Calculate contacts
# analyzer.calculate_contacts(distance_threshold=6.0)
# 
# # Generate all plots
# analyzer.plot_sequence_proximity(save_path='epitope_map.png')
# analyzer.plot_contact_map(save_path='contact_map.png')
# 
# # Save results
# analyzer.save_contact_table('contacts.csv')
# analyzer.print_summary()

print("# Uncomment the code above to analyze your own structures")

## Cleanup

In [ ]:
# Remove temporary sample files
import glob
temp_files = (['sample_complex.pdb', 'ensemble.pdb'] + 
              glob.glob('ensemble_model_*.pdb'))

for f in temp_files:
    if os.path.exists(f):
        os.remove(f)
        
print("Cleanup complete. Output files remain in outputs/")